# Stage 4: T2 Specialist 與 Cue Calibration

讀取 Stage 3 artifacts，訓練 verification timeline specialist，並進行 cue calibration 與保守版搜尋。

預設 Drive root：`/content/drive/MyDrive/VeriPromiseESG2026`。


## 0. Colab 執行環境與路徑檢查

請先執行下一個 setup cell。它會安裝套件、掛載 Google Drive、檢查 GPU、建立輸出資料夾，並驗證必要輸入檔是否存在。


In [ ]:
# Colab 啟動區塊：安裝套件、掛載 Drive、檢查 GPU、驗證必要輸入檔。
!pip -q install transformers accelerate scikit-learn pandas numpy tqdm openpyxl

from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"
OUTPUT_ROOT = f"{BASE_DIR}/outputs"
STAGE_NAME = "stage4_t2_specialist_calibration"
STAGE_DISPLAY_NAME = "Stage 4 - T2 Specialist 與 Cue Calibration"
OUTPUT_DIR = f"{OUTPUT_ROOT}/{STAGE_NAME}"
OUT_DIR = OUTPUT_DIR
os.makedirs(OUTPUT_DIR, exist_ok=True)

REQUIRED_RELATIVE_PATHS = [
    "data/vpesg4k_train_1000.json",
    "data/vpesg4k_valdata_1000.json",
    "data/vpesg4k_testdata_2000.json",
    "outputs/stage3_tta_t4_specialist_blend/stage3_t4_specialist_probs.pkl",
    "outputs/stage3_tta_t4_specialist_blend/stage3_3view_probs.pkl",
    "outputs/stage3_tta_t4_specialist_blend/stage3_best_config.json",
]

def stage_log(label, value):
    print(f"[{STAGE_NAME}] {label}: {value}")

def require_files(relative_paths):
    missing = []
    for rel in relative_paths:
        path = f"{BASE_DIR}/{rel}"
        if not os.path.exists(path):
            missing.append(path)
    if missing:
        msg = "缺少必要輸入檔。請先執行前一個 Stage，或將資料放到 Google Drive：\n" + "\n".join(missing)
        raise FileNotFoundError(msg)

require_files(REQUIRED_RELATIVE_PATHS)
stage_log("Stage", STAGE_DISPLAY_NAME)
stage_log("BASE_DIR", BASE_DIR)
stage_log("DATA_DIR", DATA_DIR)
stage_log("OUTPUT_DIR", OUTPUT_DIR)

# 完整訓練建議使用 Colab A100 或同級 GPU。
try:
    gpu_names = !nvidia-smi --query-gpu=name --format=csv,noheader
    gpu_names = list(gpu_names)
    stage_log("GPU", gpu_names)
    if not any("A100" in str(name) for name in gpu_names):
        stage_log("WARNING", "建議使用 A100；非 A100 runtime 可能需要更長時間。")
except Exception as exc:
    stage_log("WARNING", f"無法取得 GPU 資訊，請確認 Colab runtime 已啟用 GPU。{exc}")


## 輸入輸出契約

Stage 顯示名稱：`Stage 4 - T2 Specialist 與 Cue Calibration`

Google Drive 輸出資料夾：

```text
/content/drive/MyDrive/VeriPromiseESG2026/outputs/stage4_t2_specialist_calibration/
```

必要輸入檔：

- `data/vpesg4k_train_1000.json`
- `data/vpesg4k_valdata_1000.json`
- `data/vpesg4k_testdata_2000.json`
- `outputs/stage3_tta_t4_specialist_blend/stage3_t4_specialist_probs.pkl`
- `outputs/stage3_tta_t4_specialist_blend/stage3_3view_probs.pkl`
- `outputs/stage3_tta_t4_specialist_blend/stage3_best_config.json`

主要輸出檔：

- `stage4_t2_specialist_probs.pkl`
- `stage4_t2_best_config.json`
- `stage4_conservative_config.json`
- `stage4_config.json`
- `stage4_best_val_test2000_submission.csv`
- `stage4_safe_test2000_submission.csv`

若必要輸入不存在，第一個 setup cell 會直接列出缺少的路徑並停止。


## Stage 主要流程

本節補強 verification_timeline，並動態過濾 optional base choice，避免缺少候選來源時中斷。


## 1. 路徑與全域設定

In [ ]:

import os
import json
import pickle
import random
import re
import math
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V31_DIR = f"{BASE_DIR}/outputs/stage2_ckip_taskwise_ensemble"
V31_CONFIG_PATH = f"{V31_DIR}/stage2_best_val_config.json"

V32_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V32_T4_PROB_PATH = f"{V32_DIR}/stage3_t4_specialist_probs.pkl"

V33_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V33_VIEW_PROB_PATH = f"{V33_DIR}/stage3_3view_probs.pkl"

V34_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V34_CONFIG_PATH = f"{V34_DIR}/stage3_best_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
os.makedirs(OUT_DIR, exist_ok=True)

T2_PROB_PATH = f"{OUT_DIR}/stage4_t2_specialist_probs.pkl"

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

T2_LABELS = ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years"]
T4_LABELS = ["Clear", "Not Clear", "Misleading"]

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

T2_LABEL2ID = {lab: i for i, lab in enumerate(T2_LABELS)}
T2_ID2LABEL = {i: lab for lab, i in T2_LABEL2ID.items()}

T4_LABEL2ID = {lab: i for i, lab in enumerate(T4_LABELS)}
T4_ID2LABEL = {i: lab for lab, i in T4_LABEL2ID.items()}

MODEL_NAME = "hfl/chinese-macbert-base"

N_SPLITS = 5
MAX_LEN = 384
EPOCHS = 4
BATCH_SIZE = 8
PRED_BATCH_SIZE = 16
GRAD_ACCUM = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2
SEED = 45

# T2 specialist 可用 head + tail 作為訓練增強。
# 如果想更保守，可改成 ["head"]。
TRAIN_VIEWS = ["head", "tail"]
PRED_VIEWS = ["head", "middle", "tail"]

USE_CLASS_WEIGHTS = True
CLASS_WEIGHT_MAX = 8.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)
print("OUT_DIR:", OUT_DIR)
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, V33_VIEW_PROB_PATH, V32_T4_PROB_PATH, V34_CONFIG_PATH]:
    print(os.path.exists(p), p)


In [ ]:

def seed_everything(seed=45):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)


## 2. 讀取資料與 Stage3C base 快取

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(V34_CONFIG_PATH, "r", encoding="utf-8") as f:
    V34_CONFIG_OBJ = json.load(f)

V34_BASE_CONFIG = V34_CONFIG_OBJ["base_config"]
V34_T4_ALPHA = float(V34_CONFIG_OBJ["t4_alpha"])
V34_T4_SPEC_MULT = V34_CONFIG_OBJ["t4_spec_multipliers"]

with open(V33_VIEW_PROB_PATH, "rb") as f:
    view_obj = pickle.load(f)

with open(V32_T4_PROB_PATH, "rb") as f:
    t4_obj = pickle.load(f)

t4_val_probs = t4_obj["val_probs"]
t4_test_probs = t4_obj["test_probs"]

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\nStage3C_BASE_CONFIG:")
print(json.dumps(V34_BASE_CONFIG, ensure_ascii=False, indent=2))
print("\nStage3C_T4_ALPHA:", V34_T4_ALPHA)
print("Stage3C_T4_SPEC_MULT:", V34_T4_SPEC_MULT)

print("\nT4 probs:", t4_val_probs.shape, t4_test_probs.shape)

print("\nT2 train distribution among promise_status=Yes:")
t2_train_df = train_df[
    (train_df["promise_status"] == "Yes") &
    (train_df["verification_timeline"].isin(T2_LABELS))
].reset_index(drop=True)
print(t2_train_df["verification_timeline"].value_counts().to_dict())
print("t2_train_df:", t2_train_df.shape)


## 3. 評分、階層規則、Stage3C base 重建工具

In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred


In [ ]:

VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},

    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},

    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

def mix_views(stem_view_probs, split, task, preset_name):
    preset = VIEW_PRESETS[preset_name]
    out = None
    for view, w in preset.items():
        p = stem_view_probs[split][view][task]
        out = w * p if out is None else out + w * p
    return out

def build_tta_probs(split, config):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    view_presets = config.get("view_presets", {t: "head" for t in TASKS})

    out = {}
    for t in TASKS:
        preset_name = view_presets.get(t, "head")
        mac_p = mix_views(view_obj["mac"], split, t, preset_name)
        ckip_p = mix_views(view_obj["ckip"], split, t, preset_name)

        w_mac = weights.get(t, 0.5)
        p = w_mac * mac_p + (1 - w_mac) * ckip_p
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p.astype(np.float32)

    return out

def eval_config_as_pred(df, split, config):
    probs = build_tta_probs(split, config)
    post_cfg = {k: v for k, v in config.items() if k not in ["weights", "view_presets"]}
    pred = probs_to_pred_df(df, probs, post_cfg)
    return pred, probs

def t4_adjust_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T4_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_quality_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    q = adjusted["evidence_quality"]

    non_na_ids = [
        LABEL2ID["evidence_quality"]["Clear"],
        LABEL2ID["evidence_quality"]["Not Clear"],
        LABEL2ID["evidence_quality"]["Misleading"],
    ]

    q3 = q[:, non_na_ids].copy()
    q3 = q3 / np.clip(q3.sum(axis=1, keepdims=True), 1e-12, None)
    return q3

def compose_with_t4_blend(
    df,
    base_pred,
    base_probs,
    base_config,
    spec_probs,
    alpha=0.05,
    spec_multipliers=None,
):
    out = base_pred.copy()

    mask = out["evidence_status"] == "Yes"

    base_q3 = get_base_quality_non_na_probs(base_probs, base_config)
    spec_q3 = t4_adjust_probs(spec_probs, spec_multipliers=spec_multipliers)

    blend = (1 - alpha) * base_q3 + alpha * spec_q3
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)

    ids = blend.argmax(axis=1)
    labels = np.array([T4_ID2LABEL[int(i)] for i in ids])

    out.loc[mask, "evidence_quality"] = labels[mask.values]
    out.loc[~mask, "evidence_quality"] = "N/A"
    out = apply_hierarchy_to_pred_df(out)

    return out

def build_v34_base_pred_probs(df, split):
    base_pred, base_probs = eval_config_as_pred(df, split, V34_BASE_CONFIG)
    t4_probs = t4_val_probs if split == "val" else t4_test_probs

    v34_pred = compose_with_t4_blend(
        df=df,
        base_pred=base_pred,
        base_probs=base_probs,
        base_config=V34_BASE_CONFIG,
        spec_probs=t4_probs,
        alpha=V34_T4_ALPHA,
        spec_multipliers=V34_T4_SPEC_MULT,
    )
    return v34_pred, base_probs

v34_val_pred, v34_val_base_probs = build_v34_base_pred_probs(val_df, "val")
v34_metrics = calc_metrics(val_df, v34_val_pred)
print_metrics(v34_metrics, "Stage3C reconstructed base")
for t in TASKS:
    print(t, v34_val_pred[t].value_counts().to_dict())


## 4. T2 specialist model / dataset

In [ ]:

class T2SpecialistModel(nn.Module):
    def __init__(self, model_name, num_labels=4, dropout=0.2):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden * 2, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        last_hidden = out.last_hidden_state

        cls_vec = last_hidden[:, 0, :]
        mask = attention_mask.unsqueeze(-1).float()
        mean_vec = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)

        pooled = torch.cat([cls_vec, mean_vec], dim=-1)
        logits = self.classifier(self.dropout(pooled))
        return logits

class T2Dataset(Dataset):
    def __init__(self, df, tokenizer, labels=None, views=None, max_len=384):
        self.tokenizer = tokenizer
        self.max_len = max_len

        self.cls_id = getattr(tokenizer, "cls_token_id", None)
        self.sep_id = getattr(tokenizer, "sep_token_id", None)
        self.pad_id = getattr(tokenizer, "pad_token_id", None)

        if self.cls_id is None:
            self.cls_id = tokenizer.convert_tokens_to_ids("[CLS]")
        if self.sep_id is None:
            self.sep_id = tokenizer.convert_tokens_to_ids("[SEP]")
        if self.pad_id is None:
            self.pad_id = 0

        self.special_n = 2

        if views is None:
            views = ["head"]

        self.records = []
        df = df.reset_index(drop=True)

        for i, row in df.iterrows():
            text = str(row["data"])
            label = None
            if labels is not None:
                label = int(labels[i])
            for view in views:
                self.records.append((text, label, view))

    def __len__(self):
        return len(self.records)

    def select_tokens(self, token_ids, view):
        max_content_len = self.max_len - self.special_n

        if len(token_ids) <= max_content_len:
            return token_ids

        if view == "head":
            return token_ids[:max_content_len]

        if view == "tail":
            return token_ids[-max_content_len:]

        if view == "middle":
            start = max((len(token_ids) - max_content_len) // 2, 0)
            return token_ids[start:start + max_content_len]

        raise ValueError(f"unknown view: {view}")

    def __getitem__(self, idx):
        text, label, view = self.records[idx]

        token_ids = self.tokenizer.encode(text, add_special_tokens=False)
        token_ids = self.select_tokens(token_ids, view)

        input_ids = [self.cls_id] + token_ids + [self.sep_id]
        attention_mask = [1] * len(input_ids)
        token_type_ids = [0] * len(input_ids)

        pad_len = self.max_len - len(input_ids)
        if pad_len > 0:
            input_ids += [self.pad_id] * pad_len
            attention_mask += [0] * pad_len
            token_type_ids += [0] * pad_len
        else:
            input_ids = input_ids[:self.max_len]
            attention_mask = attention_mask[:self.max_len]
            token_type_ids = token_type_ids[:self.max_len]

        item = {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
        }

        if label is not None:
            item["labels"] = torch.tensor(label, dtype=torch.long)

        return item

def batch_to_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

def make_class_weights(labels):
    counts = Counter(labels)
    total = len(labels)
    weights = []
    for i, lab in T2_ID2LABEL.items():
        c = counts.get(i, 1)
        w = total / (len(T2_LABELS) * c)
        weights.append(w)
    weights = np.array(weights, dtype=np.float32)
    weights = np.clip(weights, 1.0, CLASS_WEIGHT_MAX)
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)


## 5. 訓練 T2 conditional specialist / 或載入快取

In [ ]:

def train_one_fold(fold, train_part, valid_part, tokenizer):
    print(f"\n===== Fold {fold} =====")

    train_labels = train_part["verification_timeline"].map(T2_LABEL2ID).values.astype(int)
    valid_labels = valid_part["verification_timeline"].map(T2_LABEL2ID).values.astype(int)

    train_ds = T2Dataset(train_part, tokenizer, labels=train_labels, views=TRAIN_VIEWS, max_len=MAX_LEN)
    valid_ds = T2Dataset(valid_part, tokenizer, labels=valid_labels, views=["head"], max_len=MAX_LEN)

    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    valid_dl = DataLoader(valid_ds, batch_size=PRED_BATCH_SIZE, shuffle=False, num_workers=0)

    model = T2SpecialistModel(MODEL_NAME, num_labels=len(T2_LABELS), dropout=0.2).to(DEVICE)

    if USE_CLASS_WEIGHTS:
        class_weights = make_class_weights(train_labels)
        print("class_weights:", {T2_ID2LABEL[i]: float(class_weights[i].detach().cpu()) for i in range(len(T2_LABELS))})
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    total_steps = math.ceil(len(train_dl) / GRAD_ACCUM) * EPOCHS
    warmup_steps = int(total_steps * 0.1)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))

    best_score = -1
    best_path = f"{OUT_DIR}/t2_fold{fold}_best.pt"
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        optimizer.zero_grad(set_to_none=True)

        pbar = tqdm(train_dl, desc=f"fold {fold} epoch {epoch}", leave=False)
        for step, batch in enumerate(pbar, start=1):
            batch = batch_to_device(batch)
            labels = batch.pop("labels")

            with torch.cuda.amp.autocast(enabled=(DEVICE == "cuda")):
                logits = model(**batch)
                loss = criterion(logits, labels)
                loss = loss / GRAD_ACCUM

            scaler.scale(loss).backward()

            if step % GRAD_ACCUM == 0 or step == len(train_dl):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

            losses.append(float(loss.detach().cpu()) * GRAD_ACCUM)
            pbar.set_postfix(loss=np.mean(losses[-20:]))

        valid_probs = predict_t2_probs_for_model(model, valid_part, tokenizer, view="head", batch_size=PRED_BATCH_SIZE)
        valid_pred = valid_probs.argmax(axis=1)
        score = f1_score(valid_labels, valid_pred, labels=list(range(len(T2_LABELS))), average="macro", zero_division=0)

        print(f"fold {fold} epoch {epoch}: loss={np.mean(losses):.4f} val_t2_macro_f1={score:.6f}")

        if score > best_score:
            best_score = score
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
            print("saved:", best_path)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("early stop")
                break

    del model
    torch.cuda.empty_cache()

    return best_path, best_score

@torch.no_grad()
def predict_t2_probs_for_model(model, df, tokenizer, view="head", batch_size=16):
    model.eval()
    ds = T2Dataset(df, tokenizer, labels=None, views=[view], max_len=MAX_LEN)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

    all_probs = []

    for batch in tqdm(dl, desc=f"predict t2 {view}", leave=False):
        batch = batch_to_device(batch)
        logits = model(**batch)
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        all_probs.append(probs)

    return np.concatenate(all_probs, axis=0).astype(np.float32)

def average_np_probs(list_probs):
    return np.mean(list_probs, axis=0).astype(np.float32)

if os.path.exists(T2_PROB_PATH):
    print("Loading existing T2 specialist probs:", T2_PROB_PATH)
    with open(T2_PROB_PATH, "rb") as f:
        t2_obj = pickle.load(f)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    y = t2_train_df["verification_timeline"].map(T2_LABEL2ID).values.astype(int)
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    fold_paths = []
    fold_scores = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(t2_train_df, y), start=1):
        train_part = t2_train_df.iloc[tr_idx].reset_index(drop=True)
        valid_part = t2_train_df.iloc[va_idx].reset_index(drop=True)

        path, score = train_one_fold(fold, train_part, valid_part, tokenizer)
        fold_paths.append(path)
        fold_scores.append(score)

    print("fold_scores:", fold_scores)
    print("mean fold score:", np.mean(fold_scores))

    # Predict val/test for head/middle/tail
    t2_view_probs = {
        "val": {},
        "test": {},
    }

    for view in PRED_VIEWS:
        print(f"\n=== T2 specialist predict view: {view} ===")
        val_fold_probs = []
        test_fold_probs = []

        for fold, path in enumerate(fold_paths, start=1):
            print("loading", path)
            model = T2SpecialistModel(MODEL_NAME, num_labels=len(T2_LABELS), dropout=0.2).to(DEVICE)
            state = torch.load(path, map_location=DEVICE)
            model.load_state_dict(state)

            val_probs = predict_t2_probs_for_model(model, val_df, tokenizer, view=view, batch_size=PRED_BATCH_SIZE)
            test_probs = predict_t2_probs_for_model(model, test_df, tokenizer, view=view, batch_size=PRED_BATCH_SIZE)

            val_fold_probs.append(val_probs)
            test_fold_probs.append(test_probs)

            del model
            torch.cuda.empty_cache()

        t2_view_probs["val"][view] = average_np_probs(val_fold_probs)
        t2_view_probs["test"][view] = average_np_probs(test_fold_probs)

    t2_obj = {
        "model_name": MODEL_NAME,
        "fold_paths": fold_paths,
        "fold_scores": fold_scores,
        "train_views": TRAIN_VIEWS,
        "pred_views": PRED_VIEWS,
        "labels": T2_LABELS,
        "val_probs": t2_view_probs["val"],
        "test_probs": t2_view_probs["test"],
        "val_ids": val_df["id"].tolist(),
        "test_ids": test_df["id"].tolist(),
    }

    with open(T2_PROB_PATH, "wb") as f:
        pickle.dump(t2_obj, f)
    print("saved:", T2_PROB_PATH)

print("T2 object keys:", t2_obj.keys())
print("fold_scores:", t2_obj.get("fold_scores"))
for split in ["val", "test"]:
    for view in PRED_VIEWS:
        print(split, view, t2_obj[f"{split}_probs"][view].shape)


## 6. T2 specialist 接回 Stage3C base

In [ ]:

T2_VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},
    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
}

def mix_t2_views(split, preset_name):
    preset = T2_VIEW_PRESETS[preset_name]
    prob_dict = t2_obj[f"{split}_probs"]

    out = None
    for view, w in preset.items():
        p = prob_dict[view]
        out = w * p if out is None else out + w * p

    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def adjust_t2_spec_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T2_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_timeline_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    p = adjusted["verification_timeline"]

    non_na_ids = [
        LABEL2ID["verification_timeline"]["already"],
        LABEL2ID["verification_timeline"]["within_2_years"],
        LABEL2ID["verification_timeline"]["between_2_and_5_years"],
        LABEL2ID["verification_timeline"]["more_than_5_years"],
    ]

    p4 = p[:, non_na_ids].copy()
    p4 = p4 / np.clip(p4.sum(axis=1, keepdims=True), 1e-12, None)
    return p4

def compose_with_t2_blend(
    df,
    base_pred,
    base_probs,
    base_config,
    spec_probs,
    alpha=0.10,
    spec_multipliers=None,
):
    out = base_pred.copy()

    # T2 specialist 只在 promise_status=Yes 的樣本生效
    mask = out["promise_status"] == "Yes"

    base_p4 = get_base_timeline_non_na_probs(base_probs, base_config)
    spec_p4 = adjust_t2_spec_probs(spec_probs, spec_multipliers=spec_multipliers)

    blend = (1 - alpha) * base_p4 + alpha * spec_p4
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)

    ids = blend.argmax(axis=1)
    labels = np.array([T2_ID2LABEL[int(i)] for i in ids])

    out.loc[mask, "verification_timeline"] = labels[mask.values]
    out.loc[~mask, "verification_timeline"] = "N/A"

    out = apply_hierarchy_to_pred_df(out)
    return out

def eval_v35_combo(split, df, alpha, t2_view_preset, spec_multipliers):
    base_pred, base_probs = build_v34_base_pred_probs(df, split)
    spec_probs = mix_t2_views(split, t2_view_preset)

    pred = compose_with_t2_blend(
        df=df,
        base_pred=base_pred,
        base_probs=base_probs,
        base_config=V34_BASE_CONFIG,
        spec_probs=spec_probs,
        alpha=alpha,
        spec_multipliers=spec_multipliers,
    )

    return pred, base_pred, base_probs, spec_probs


## 7. val1000 搜尋：alpha / view / class multipliers

In [ ]:

# 先看直接 replace 是否有用
replace_pred, _, _, _ = eval_v35_combo(
    split="val",
    df=val_df,
    alpha=1.0,
    t2_view_preset="tail",
    spec_multipliers=None,
)
replace_metrics = calc_metrics(val_df, replace_pred)
print_metrics(replace_metrics, "Stage4A T2 specialist replace tail")

# light search
alpha_values = [0.0, 0.02, 0.03, 0.05, 0.07, 0.10, 0.15, 0.20, 0.30, 0.50, 1.0]
view_values = ["head", "tail", "head_tail", "tail_more", "head_mid_tail", "mid_tail", "even", "mid_more"]

spec_mult_candidates = [
    None,
    {"already": 1.0, "within_2_years": 1.0, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 1.0, "within_2_years": 0.5, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 1.0, "within_2_years": 0.75, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 1.0, "within_2_years": 1.5, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 1.0, "within_2_years": 2.0, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 0.9, "within_2_years": 1.5, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 1.0, "within_2_years": 1.0, "between_2_and_5_years": 1.1, "more_than_5_years": 1.1},
    {"already": 0.9, "within_2_years": 1.0, "between_2_and_5_years": 1.1, "more_than_5_years": 1.1},
]

rows = []
best = None

for view_name in view_values:
    for alpha in alpha_values:
        for spec_mult in spec_mult_candidates:
            pred, _, _, _ = eval_v35_combo(
                split="val",
                df=val_df,
                alpha=alpha,
                t2_view_preset=view_name,
                spec_multipliers=spec_mult,
            )
            metrics = calc_metrics(val_df, pred)
            score = metrics["weighted_score"]

            row = {
                "view": view_name,
                "alpha": alpha,
                "spec_mult": json.dumps(spec_mult, ensure_ascii=False),
                **metrics,
                "pred_already": int((pred["verification_timeline"] == "already").sum()),
                "pred_within_2": int((pred["verification_timeline"] == "within_2_years").sum()),
                "pred_2_5": int((pred["verification_timeline"] == "between_2_and_5_years").sum()),
                "pred_more_5": int((pred["verification_timeline"] == "more_than_5_years").sum()),
                "pred_NA": int((pred["verification_timeline"] == "N/A").sum()),
            }
            rows.append(row)

            if best is None or score > best[0]:
                best = (score, metrics, pred, view_name, alpha, spec_mult)

search_df = pd.DataFrame(rows).sort_values("weighted_score", ascending=False)
search_df.to_csv(f"{OUT_DIR}/stage4_t2_blend_search.csv", index=False, encoding="utf-8-sig")

best_score, best_metrics, best_pred, best_view, best_alpha, best_spec_mult = best

print_metrics(v34_metrics, "Stage3C base")
print_metrics(replace_metrics, "Stage4A replace")
print_metrics(best_metrics, "BEST Stage4A val1000")

print("best_view:", best_view)
print("best_alpha:", best_alpha)
print("best_spec_mult:", best_spec_mult)

for t in TASKS:
    print("\n", t)
    print(best_pred[t].value_counts().to_dict())

display(search_df.head(30))


## 8. 儲存 val1000 錯誤分析與 config

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
        ],
        axis=1,
    )

    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]

    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

best_val_full = save_val_outputs(best_pred, best_metrics, "stage4_t2_best_val")
replace_val_full = save_val_outputs(replace_pred, replace_metrics, "stage4_t2_replace")
base_val_full = save_val_outputs(v34_val_pred, v34_metrics, "stage4_t2_base_stage3")

v35_config_obj = {
    "base": "stage3_best_val",
    "model_name": MODEL_NAME,
    "t2_labels": T2_LABELS,
    "train_views": TRAIN_VIEWS,
    "pred_views": PRED_VIEWS,
    "fold_scores": t2_obj.get("fold_scores"),
    "best_view": best_view,
    "best_alpha": best_alpha,
    "best_spec_multipliers": best_spec_mult,
    "v34_metrics": v34_metrics,
    "replace_metrics": replace_metrics,
    "best_metrics": best_metrics,
}

with open(f"{OUT_DIR}/stage4_t2_best_config.json", "w", encoding="utf-8") as f:
    json.dump(v35_config_obj, f, ensure_ascii=False, indent=2)

print("saved config and val outputs to:", OUT_DIR)
display(best_val_full["any_error"].value_counts())
display(best_val_full.head())


## 9. 產生 test2000 submissions

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def make_submission_from_pred(pred, tag):
    pred = pred.copy()

    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(pred)
    sub = pred[["id"] + TASKS].copy()

    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

# base Stage3C
v34_test_pred, _ = build_v34_base_pred_probs(test_df, "test")
base_sub, base_path = make_submission_from_pred(v34_test_pred, "stage4_t2_base_stage3")

# replace tail
replace_test_pred, _, _, _ = eval_v35_combo(
    split="test",
    df=test_df,
    alpha=1.0,
    t2_view_preset="tail",
    spec_multipliers=None,
)
replace_sub, replace_path = make_submission_from_pred(replace_test_pred, "stage4_t2_replace")

# best blend
best_test_pred, _, _, _ = eval_v35_combo(
    split="test",
    df=test_df,
    alpha=best_alpha,
    t2_view_preset=best_view,
    spec_multipliers=best_spec_mult,
)
best_sub, best_path = make_submission_from_pred(best_test_pred, "stage4_t2_best_blend")

# conservative：alpha 不超過 0.10，且降低 within_2_years multiplier
conservative_alpha = min(best_alpha, 0.10)
conservative_mult = None if best_spec_mult is None else json.loads(json.dumps(best_spec_mult))
if conservative_mult is not None and "within_2_years" in conservative_mult:
    conservative_mult["within_2_years"] = min(float(conservative_mult["within_2_years"]), 1.0)

conservative_test_pred, _, _, _ = eval_v35_combo(
    split="test",
    df=test_df,
    alpha=conservative_alpha,
    t2_view_preset=best_view,
    spec_multipliers=conservative_mult,
)
conservative_sub, conservative_path = make_submission_from_pred(conservative_test_pred, "stage4_t2_conservative")

print("\nSubmission paths:")
print(base_path)
print(replace_path)
print(best_path)
print(conservative_path)


## 10. 報告摘要

In [ ]:

report = []
report.append("# Stage4A T2 conditional specialist report")
report.append("")
report.append("## Stage3C base metrics")
for k, v in v34_metrics.items():
    report.append(f"- {k}: {v:.6f}")

report.append("")
report.append("## T2 specialist replace metrics")
for k, v in replace_metrics.items():
    report.append(f"- {k}: {v:.6f}")

report.append("")
report.append("## Best Stage4A metrics")
for k, v in best_metrics.items():
    report.append(f"- {k}: {v:.6f}")

report.append("")
report.append("## Best config")
report.append("```json")
report.append(json.dumps(v35_config_obj, ensure_ascii=False, indent=2))
report.append("```")

report.append("")
report.append("## Test distribution: stage4_t2_best_blend")
for t in TASKS:
    report.append(f"### {t}")
    for lab, cnt in best_sub[t].value_counts().to_dict().items():
        report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)

with open(f"{OUT_DIR}/stage4_t2_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:5000])
print("saved:", f"{OUT_DIR}/stage4_t2_report.md")


## Stage 4B conservative T2 blend search

This section uses public Stage paths and artifact names under `/content/drive/MyDrive/VeriPromiseESG2026`.


## 1. 路徑與設定

In [ ]:

import os
import json
import pickle
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V32_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V32_T4_PROB_PATH = f"{V32_DIR}/stage3_t4_specialist_probs.pkl"

V33_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V33_VIEW_PROB_PATH = f"{V33_DIR}/stage3_3view_probs.pkl"

V34_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V34_CONFIG_PATH = f"{V34_DIR}/stage3_best_config.json"

V35_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V35_T2_PROB_PATH = f"{V35_DIR}/stage4_t2_specialist_probs.pkl"
V35_CONFIG_PATH = f"{V35_DIR}/stage4_t2_best_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
os.makedirs(OUT_DIR, exist_ok=True)

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

T2_LABELS = ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years"]
T4_LABELS = ["Clear", "Not Clear", "Misleading"]

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

T2_LABEL2ID = {lab: i for i, lab in enumerate(T2_LABELS)}
T2_ID2LABEL = {i: lab for lab, i in T2_LABEL2ID.items()}

T4_LABEL2ID = {lab: i for i, lab in enumerate(T4_LABELS)}
T4_ID2LABEL = {i: lab for lab, i in T4_LABEL2ID.items()}

print("OUT_DIR:", OUT_DIR)
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, V33_VIEW_PROB_PATH, V32_T4_PROB_PATH, V34_CONFIG_PATH, V35_T2_PROB_PATH]:
    print(os.path.exists(p), p)


## 2. 讀取資料與快取

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(V34_CONFIG_PATH, "r", encoding="utf-8") as f:
    V34_CONFIG_OBJ = json.load(f)

V34_BASE_CONFIG = V34_CONFIG_OBJ["base_config"]
V34_T4_ALPHA = float(V34_CONFIG_OBJ["t4_alpha"])
V34_T4_SPEC_MULT = V34_CONFIG_OBJ["t4_spec_multipliers"]

with open(V33_VIEW_PROB_PATH, "rb") as f:
    view_obj = pickle.load(f)

with open(V32_T4_PROB_PATH, "rb") as f:
    t4_obj = pickle.load(f)

t4_val_probs = t4_obj["val_probs"]
t4_test_probs = t4_obj["test_probs"]

with open(V35_T2_PROB_PATH, "rb") as f:
    t2_obj = pickle.load(f)

if os.path.exists(V35_CONFIG_PATH):
    with open(V35_CONFIG_PATH, "r", encoding="utf-8") as f:
        V35_CONFIG_OBJ = json.load(f)
else:
    V35_CONFIG_OBJ = None

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\nStage3C_BASE_CONFIG:")
print(json.dumps(V34_BASE_CONFIG, ensure_ascii=False, indent=2))

print("\nStage3C_T4_ALPHA:", V34_T4_ALPHA)
print("Stage3C_T4_SPEC_MULT:", V34_T4_SPEC_MULT)

print("\nT2 obj keys:", t2_obj.keys())
print("T2 fold scores:", t2_obj.get("fold_scores"))
for split in ["val", "test"]:
    for view in ["head", "middle", "tail"]:
        print(split, view, t2_obj[f"{split}_probs"][view].shape)


## 3. 評分與後處理工具

In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred


## 4. 重建 Stage3C base

In [ ]:

VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},

    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},

    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

def mix_views(stem_view_probs, split, task, preset_name):
    preset = VIEW_PRESETS[preset_name]
    out = None
    for view, w in preset.items():
        p = stem_view_probs[split][view][task]
        out = w * p if out is None else out + w * p
    return out

def build_tta_probs(split, config):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    view_presets = config.get("view_presets", {t: "head" for t in TASKS})

    out = {}
    for t in TASKS:
        preset_name = view_presets.get(t, "head")
        mac_p = mix_views(view_obj["mac"], split, t, preset_name)
        ckip_p = mix_views(view_obj["ckip"], split, t, preset_name)

        w_mac = weights.get(t, 0.5)
        p = w_mac * mac_p + (1 - w_mac) * ckip_p
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p.astype(np.float32)

    return out

def eval_config_as_pred(df, split, config):
    probs = build_tta_probs(split, config)
    post_cfg = {k: v for k, v in config.items() if k not in ["weights", "view_presets"]}
    pred = probs_to_pred_df(df, probs, post_cfg)
    return pred, probs

def t4_adjust_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T4_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_quality_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    q = adjusted["evidence_quality"]

    non_na_ids = [
        LABEL2ID["evidence_quality"]["Clear"],
        LABEL2ID["evidence_quality"]["Not Clear"],
        LABEL2ID["evidence_quality"]["Misleading"],
    ]

    q3 = q[:, non_na_ids].copy()
    q3 = q3 / np.clip(q3.sum(axis=1, keepdims=True), 1e-12, None)
    return q3

def compose_with_t4_blend(df, base_pred, base_probs, base_config, spec_probs, alpha=0.05, spec_multipliers=None):
    out = base_pred.copy()
    mask = out["evidence_status"] == "Yes"

    base_q3 = get_base_quality_non_na_probs(base_probs, base_config)
    spec_q3 = t4_adjust_probs(spec_probs, spec_multipliers=spec_multipliers)

    blend = (1 - alpha) * base_q3 + alpha * spec_q3
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)

    ids = blend.argmax(axis=1)
    labels = np.array([T4_ID2LABEL[int(i)] for i in ids])

    out.loc[mask, "evidence_quality"] = labels[mask.values]
    out.loc[~mask, "evidence_quality"] = "N/A"
    out = apply_hierarchy_to_pred_df(out)

    return out

def build_v34_base_pred_probs(df, split):
    base_pred, base_probs = eval_config_as_pred(df, split, V34_BASE_CONFIG)
    t4_probs = t4_val_probs if split == "val" else t4_test_probs

    v34_pred = compose_with_t4_blend(
        df=df,
        base_pred=base_pred,
        base_probs=base_probs,
        base_config=V34_BASE_CONFIG,
        spec_probs=t4_probs,
        alpha=V34_T4_ALPHA,
        spec_multipliers=V34_T4_SPEC_MULT,
    )
    return v34_pred, base_probs

v34_val_pred, v34_val_base_probs = build_v34_base_pred_probs(val_df, "val")
v34_metrics = calc_metrics(val_df, v34_val_pred)
print_metrics(v34_metrics, "Stage3C reconstructed base")
for t in TASKS:
    print(t, v34_val_pred[t].value_counts().to_dict())


## 5. T2 blend 工具

In [ ]:

T2_VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},
    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
}

def mix_t2_views(split, preset_name):
    preset = T2_VIEW_PRESETS[preset_name]
    prob_dict = t2_obj[f"{split}_probs"]

    out = None
    for view, w in preset.items():
        p = prob_dict[view]
        out = w * p if out is None else out + w * p

    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def adjust_t2_spec_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T2_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_timeline_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    p = adjusted["verification_timeline"]

    non_na_ids = [
        LABEL2ID["verification_timeline"]["already"],
        LABEL2ID["verification_timeline"]["within_2_years"],
        LABEL2ID["verification_timeline"]["between_2_and_5_years"],
        LABEL2ID["verification_timeline"]["more_than_5_years"],
    ]

    p4 = p[:, non_na_ids].copy()
    p4 = p4 / np.clip(p4.sum(axis=1, keepdims=True), 1e-12, None)
    return p4

def compose_with_t2_blend(df, base_pred, base_probs, base_config, spec_probs, alpha=0.10, spec_multipliers=None):
    out = base_pred.copy()
    mask = out["promise_status"] == "Yes"

    base_p4 = get_base_timeline_non_na_probs(base_probs, base_config)
    spec_p4 = adjust_t2_spec_probs(spec_probs, spec_multipliers=spec_multipliers)

    blend = (1 - alpha) * base_p4 + alpha * spec_p4
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)

    ids = blend.argmax(axis=1)
    labels = np.array([T2_ID2LABEL[int(i)] for i in ids])

    out.loc[mask, "verification_timeline"] = labels[mask.values]
    out.loc[~mask, "verification_timeline"] = "N/A"

    out = apply_hierarchy_to_pred_df(out)
    return out

def eval_stage4_conservative_combo(split, df, alpha, t2_view_preset, spec_multipliers):
    base_pred, base_probs = build_v34_base_pred_probs(df, split)
    spec_probs = mix_t2_views(split, t2_view_preset)

    pred = compose_with_t2_blend(
        df=df,
        base_pred=base_pred,
        base_probs=base_probs,
        base_config=V34_BASE_CONFIG,
        spec_probs=spec_probs,
        alpha=alpha,
        spec_multipliers=spec_multipliers,
    )

    return pred, base_pred, base_probs, spec_probs


## 6. Conservative Search

In [ ]:

# Stage4A best 參考：alpha=0.5, view=head_tail, within_2_years=2.0
# stage4_conservative 限制更保守

alpha_values = [0.0, 0.02, 0.03, 0.05, 0.07, 0.10, 0.12, 0.15, 0.20, 0.25, 0.30]
view_values = ["head", "head_tail", "tail", "tail_more", "head_mid_tail", "even", "mid_tail", "mid_more"]

spec_mult_candidates = [
    None,
    {"already": 1.0, "within_2_years": 1.0, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 1.0, "within_2_years": 1.25, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 1.0, "within_2_years": 1.5, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 0.95, "within_2_years": 1.25, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 0.95, "within_2_years": 1.5, "between_2_and_5_years": 1.0, "more_than_5_years": 1.0},
    {"already": 1.0, "within_2_years": 1.25, "between_2_and_5_years": 1.05, "more_than_5_years": 1.05},
    {"already": 0.95, "within_2_years": 1.25, "between_2_and_5_years": 1.05, "more_than_5_years": 1.05},
]

# risk reference from Stage3C base test distribution
v34_test_pred, _ = build_v34_base_pred_probs(test_df, "test")
v34_test_dist = v34_test_pred["verification_timeline"].value_counts().to_dict()
print("v34 test timeline reference:", v34_test_dist)

def timeline_counts(pred):
    return {lab: int((pred["verification_timeline"] == lab).sum()) for lab in LABELS["verification_timeline"]}

def calc_distribution_risk(test_pred, ref_dist):
    cnt = timeline_counts(test_pred)

    # 對 test 分布的保守風險分數：
    # - already 不要降太多
    # - more_than_5_years 不要暴增太多
    # - within_2_years 不要暴增太多
    already_drop = max(0, ref_dist.get("already", 0) - cnt.get("already", 0))
    more5_inc = max(0, cnt.get("more_than_5_years", 0) - ref_dist.get("more_than_5_years", 0))
    within2_inc = max(0, cnt.get("within_2_years", 0) - ref_dist.get("within_2_years", 0))

    risk = (
        already_drop / 120.0
        + more5_inc / 120.0
        + within2_inc / 25.0
    )
    return float(risk), {
        "already_drop": already_drop,
        "more5_inc": more5_inc,
        "within2_inc": within2_inc,
    }

rows = []
best_val = None
best_safe = None
best_very_safe = None
best_dist_matched = None

for view_name in view_values:
    for alpha in alpha_values:
        for spec_mult in spec_mult_candidates:
            val_pred, _, _, _ = eval_stage4_conservative_combo(
                split="val",
                df=val_df,
                alpha=alpha,
                t2_view_preset=view_name,
                spec_multipliers=spec_mult,
            )
            val_metrics = calc_metrics(val_df, val_pred)

            test_pred, _, _, _ = eval_stage4_conservative_combo(
                split="test",
                df=test_df,
                alpha=alpha,
                t2_view_preset=view_name,
                spec_multipliers=spec_mult,
            )
            risk, risk_parts = calc_distribution_risk(test_pred, v34_test_dist)
            test_cnt = timeline_counts(test_pred)

            row = {
                "view": view_name,
                "alpha": alpha,
                "spec_mult": json.dumps(spec_mult, ensure_ascii=False),
                **val_metrics,
                "risk": risk,
                **risk_parts,
                "test_already": test_cnt.get("already", 0),
                "test_within_2": test_cnt.get("within_2_years", 0),
                "test_2_5": test_cnt.get("between_2_and_5_years", 0),
                "test_more_5": test_cnt.get("more_than_5_years", 0),
                "test_NA": test_cnt.get("N/A", 0),
                "val_already": int((val_pred["verification_timeline"] == "already").sum()),
                "val_within_2": int((val_pred["verification_timeline"] == "within_2_years").sum()),
                "val_2_5": int((val_pred["verification_timeline"] == "between_2_and_5_years").sum()),
                "val_more_5": int((val_pred["verification_timeline"] == "more_than_5_years").sum()),
                "val_NA": int((val_pred["verification_timeline"] == "N/A").sum()),
            }
            rows.append(row)

            cfg_tuple = (val_metrics["weighted_score"], val_metrics, val_pred, test_pred, view_name, alpha, spec_mult, risk, risk_parts)

            # best_val: 單純最高 val
            if best_val is None or cfg_tuple[0] > best_val[0]:
                best_val = cfg_tuple

            # safe: 分數不低於 v34 + 0.002，且風險小
            if val_metrics["weighted_score"] >= v34_metrics["weighted_score"] + 0.002 and risk <= 0.80:
                if best_safe is None or val_metrics["weighted_score"] > best_safe[0]:
                    best_safe = cfg_tuple

            # very_safe: 分數不低於 v34 + 0.001，且風險更小
            if val_metrics["weighted_score"] >= v34_metrics["weighted_score"] + 0.001 and risk <= 0.45:
                if best_very_safe is None or val_metrics["weighted_score"] > best_very_safe[0]:
                    best_very_safe = cfg_tuple

            # distribution_matched: 在有提升的前提下，風險最小；同風險比分數
            if val_metrics["weighted_score"] >= v34_metrics["weighted_score"] + 0.001:
                if best_dist_matched is None:
                    best_dist_matched = cfg_tuple
                else:
                    if (risk < best_dist_matched[7] - 1e-9) or (abs(risk - best_dist_matched[7]) < 1e-9 and val_metrics["weighted_score"] > best_dist_matched[0]):
                        best_dist_matched = cfg_tuple

search_df = pd.DataFrame(rows).sort_values(["weighted_score", "risk"], ascending=[False, True])
search_df.to_csv(f"{OUT_DIR}/stage4_conservative_search.csv", index=False, encoding="utf-8-sig")

def show_choice(name, item):
    print("\n" + "="*80)
    print(name)
    print("="*80)
    if item is None:
        print("None")
        return
    score, metrics, val_pred, test_pred, view_name, alpha, spec_mult, risk, risk_parts = item
    print_metrics(metrics, name)
    print("view:", view_name)
    print("alpha:", alpha)
    print("spec_mult:", spec_mult)
    print("risk:", risk, risk_parts)
    print("val timeline:", timeline_counts(val_pred))
    print("test timeline:", timeline_counts(test_pred))

print_metrics(v34_metrics, "v34 base")
show_choice("stage4_conservative best_val", best_val)
show_choice("stage4_conservative safe", best_safe)
show_choice("stage4_conservative very_safe", best_very_safe)
show_choice("stage4_conservative distribution_matched", best_dist_matched)

display(search_df.head(30))
display(search_df.sort_values(["risk", "weighted_score"], ascending=[True, False]).head(30))


## 7. 儲存 val 分析與 config

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
        ],
        axis=1,
    )

    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]

    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

def item_to_config(item):
    if item is None:
        return None
    score, metrics, val_pred, test_pred, view_name, alpha, spec_mult, risk, risk_parts = item
    return {
        "view": view_name,
        "alpha": alpha,
        "spec_multipliers": spec_mult,
        "metrics": metrics,
        "risk": risk,
        "risk_parts": risk_parts,
        "val_timeline_dist": timeline_counts(val_pred),
        "test_timeline_dist": timeline_counts(test_pred),
    }

choices = {
    "best_val": best_val,
    "safe": best_safe,
    "very_safe": best_very_safe,
    "distribution_matched": best_dist_matched,
}

config_obj = {
    "base": "stage3_best_val",
    "v34_metrics": v34_metrics,
    "v34_test_timeline_reference": v34_test_dist,
    "choices": {name: item_to_config(item) for name, item in choices.items()},
}

with open(f"{OUT_DIR}/stage4_conservative_config.json", "w", encoding="utf-8") as f:
    json.dump(config_obj, f, ensure_ascii=False, indent=2)

# 儲存各選擇的 val error analysis
for name, item in choices.items():
    if item is not None:
        _, metrics, val_pred, _, _, _, _, _, _ = item
        save_val_outputs(val_pred, metrics, f"stage4_conservative_{name}")

print("saved config and val outputs to:", OUT_DIR)
print(json.dumps(config_obj, ensure_ascii=False, indent=2)[:5000])


## 8. 產生 test submissions

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def make_submission_from_pred(pred, tag):
    pred = pred.copy()

    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(pred)
    sub = pred[["id"] + TASKS].copy()

    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

# base v34
base_sub, base_path = make_submission_from_pred(v34_test_pred, "stage4_conservative_base_stage3")

submission_paths = {"base_v34": base_path}
submission_subs = {"base_v34": base_sub}

for name, item in choices.items():
    if item is None:
        continue
    _, metrics, val_pred, test_pred, view_name, alpha, spec_mult, risk, risk_parts = item
    sub, path = make_submission_from_pred(test_pred, f"stage4_conservative_{name}")
    submission_paths[name] = path
    submission_subs[name] = sub

print("\nSubmission paths:")
for name, path in submission_paths.items():
    print(name, ":", path)


## 9. 報告摘要

In [ ]:

report = []
report.append("# stage4_conservative Conservative T2 Blend Search Report")
report.append("")
report.append("## Base")
report.append("### Stage3C base metrics")
for k, v in v34_metrics.items():
    report.append(f"- {k}: {v:.6f}")

report.append("")
report.append("## Choices")
for name, item in choices.items():
    report.append(f"### {name}")
    cfg = item_to_config(item)
    if cfg is None:
        report.append("- None")
    else:
        report.append(f"- view: {cfg['view']}")
        report.append(f"- alpha: {cfg['alpha']}")
        report.append(f"- spec_multipliers: {cfg['spec_multipliers']}")
        report.append(f"- risk: {cfg['risk']:.6f}")
        for k, v in cfg["metrics"].items():
            report.append(f"- {k}: {v:.6f}")
        report.append(f"- val_timeline_dist: {cfg['val_timeline_dist']}")
        report.append(f"- test_timeline_dist: {cfg['test_timeline_dist']}")

report.append("")
report.append("## Submission distributions")
for name, sub in submission_subs.items():
    report.append(f"### {name}")
    for t in TASKS:
        report.append(f"#### {t}")
        for lab, cnt in sub[t].value_counts().to_dict().items():
            report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)

with open(f"{OUT_DIR}/stage4_conservative_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:7000])
print("saved:", f"{OUT_DIR}/stage4_conservative_report.md")


## Stage 4C cue-based calibration

This section uses public Stage paths and artifact names under `/content/drive/MyDrive/VeriPromiseESG2026`.


## 1. 路徑與設定

In [ ]:

import os
import json
import pickle
import re
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, confusion_matrix

BASE_DIR = "/content/drive/MyDrive/VeriPromiseESG2026"
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/vpesg4k_train_1000.json"
VAL_PATH   = f"{DATA_DIR}/vpesg4k_valdata_1000.json"
TEST_PATH  = f"{DATA_DIR}/vpesg4k_testdata_2000.json"

V32_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V32_T4_PROB_PATH = f"{V32_DIR}/stage3_t4_specialist_probs.pkl"

V33_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V33_VIEW_PROB_PATH = f"{V33_DIR}/stage3_3view_probs.pkl"

V34_DIR = f"{BASE_DIR}/outputs/stage3_tta_t4_specialist_blend"
V34_CONFIG_PATH = f"{V34_DIR}/stage3_best_config.json"

V35_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V35_T2_PROB_PATH = f"{V35_DIR}/stage4_t2_specialist_probs.pkl"
V35_CONFIG_PATH = f"{V35_DIR}/stage4_t2_best_config.json"

V35_01_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
V35_01_CONFIG_PATH = f"{V35_01_DIR}/stage4_conservative_config.json"

OUT_DIR = f"{BASE_DIR}/outputs/stage4_t2_specialist_calibration"
os.makedirs(OUT_DIR, exist_ok=True)

TASKS = [
    "promise_status",
    "verification_timeline",
    "evidence_status",
    "evidence_quality",
]

LABELS = {
    "promise_status": ["No", "Yes"],
    "verification_timeline": ["N/A", "already", "within_2_years", "between_2_and_5_years", "more_than_5_years"],
    "evidence_status": ["N/A", "No", "Yes"],
    "evidence_quality": ["N/A", "Clear", "Not Clear", "Misleading"],
}

T2_LABELS = ["already", "within_2_years", "between_2_and_5_years", "more_than_5_years"]
T4_LABELS = ["Clear", "Not Clear", "Misleading"]

LABEL2ID = {t: {lab: i for i, lab in enumerate(labs)} for t, labs in LABELS.items()}
ID2LABEL = {t: {i: lab for i, lab in enumerate(labs)} for t, labs in LABELS.items()}

T2_LABEL2ID = {lab: i for i, lab in enumerate(T2_LABELS)}
T2_ID2LABEL = {i: lab for lab, i in T2_LABEL2ID.items()}

T4_LABEL2ID = {lab: i for i, lab in enumerate(T4_LABELS)}
T4_ID2LABEL = {i: lab for lab, i in T4_LABEL2ID.items()}

print("OUT_DIR:", OUT_DIR)
for p in [TRAIN_PATH, VAL_PATH, TEST_PATH, V33_VIEW_PROB_PATH, V32_T4_PROB_PATH, V34_CONFIG_PATH, V35_T2_PROB_PATH, V35_CONFIG_PATH, V35_01_CONFIG_PATH]:
    print(os.path.exists(p), p)


## 2. 讀取資料與快取

In [ ]:

def load_json_df(path):
    with open(path, "r", encoding="utf-8") as f:
        return pd.DataFrame(json.load(f))

train_df = load_json_df(TRAIN_PATH)
val_df = load_json_df(VAL_PATH)
test_df = load_json_df(TEST_PATH)

for df in [train_df, val_df]:
    for t in TASKS:
        df[t] = df[t].fillna("N/A").astype(str).replace({"": "N/A", "nan": "N/A"})

with open(V34_CONFIG_PATH, "r", encoding="utf-8") as f:
    V34_CONFIG_OBJ = json.load(f)

V34_BASE_CONFIG = V34_CONFIG_OBJ["base_config"]
V34_T4_ALPHA = float(V34_CONFIG_OBJ["t4_alpha"])
V34_T4_SPEC_MULT = V34_CONFIG_OBJ["t4_spec_multipliers"]

with open(V33_VIEW_PROB_PATH, "rb") as f:
    view_obj = pickle.load(f)

with open(V32_T4_PROB_PATH, "rb") as f:
    t4_obj = pickle.load(f)

t4_val_probs = t4_obj["val_probs"]
t4_test_probs = t4_obj["test_probs"]

with open(V35_T2_PROB_PATH, "rb") as f:
    t2_obj = pickle.load(f)

with open(V35_CONFIG_PATH, "r", encoding="utf-8") as f:
    V35_CONFIG_OBJ = json.load(f)

with open(V35_01_CONFIG_PATH, "r", encoding="utf-8") as f:
    V35_01_CONFIG_OBJ = json.load(f)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

print("\nV34 weighted:", V34_CONFIG_OBJ["best_metrics"]["weighted_score"] if "best_metrics" in V34_CONFIG_OBJ else None)
print("V35 weighted:", V35_CONFIG_OBJ["best_metrics"]["weighted_score"])
print("V35_01 choices:", list(V35_01_CONFIG_OBJ["choices"].keys()))

print("\nT2 fold scores:", t2_obj.get("fold_scores"))
for split in ["val", "test"]:
    for view in ["head", "middle", "tail"]:
        print(split, view, t2_obj[f"{split}_probs"][view].shape)


## 3. 基本評分與後處理

In [ ]:

def apply_hierarchy_to_pred_df(pred_df, force_quality_non_na_when_evidence_yes=False, probs=None):
    pred_df = pred_df.copy()

    no_mask = pred_df["promise_status"] == "No"
    pred_df.loc[no_mask, "verification_timeline"] = "N/A"
    pred_df.loc[no_mask, "evidence_status"] = "N/A"
    pred_df.loc[no_mask, "evidence_quality"] = "N/A"

    not_evidence_yes = pred_df["evidence_status"] != "Yes"
    pred_df.loc[not_evidence_yes, "evidence_quality"] = "N/A"

    if force_quality_non_na_when_evidence_yes and probs is not None:
        mask = (pred_df["evidence_status"] == "Yes") & (pred_df["evidence_quality"] == "N/A")
        if mask.any():
            q_probs = probs["evidence_quality"][mask.values].copy()
            q_probs[:, LABEL2ID["evidence_quality"]["N/A"]] = -1
            ids = q_probs.argmax(axis=1)
            pred_df.loc[mask, "evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    return pred_df

def f1_yes(y_true, y_pred):
    y_true_bin = (pd.Series(y_true).astype(str).values == "Yes").astype(int)
    y_pred_bin = (pd.Series(y_pred).astype(str).values == "Yes").astype(int)
    return f1_score(y_true_bin, y_pred_bin, pos_label=1, average="binary", zero_division=0)

def macro_f1_task(y_true, y_pred, task):
    return f1_score(
        y_true,
        y_pred,
        labels=LABELS[task],
        average="macro",
        zero_division=0,
    )

def calc_metrics(y_true_df, pred_df):
    out = {}
    out["promise_status_f1_yes"] = f1_yes(y_true_df["promise_status"], pred_df["promise_status"])
    out["verification_timeline_macro_f1"] = macro_f1_task(y_true_df["verification_timeline"], pred_df["verification_timeline"], "verification_timeline")
    out["evidence_status_f1_yes"] = f1_yes(y_true_df["evidence_status"], pred_df["evidence_status"])
    out["evidence_quality_macro_f1"] = macro_f1_task(y_true_df["evidence_quality"], pred_df["evidence_quality"], "evidence_quality")
    out["weighted_score"] = (
        0.20 * out["promise_status_f1_yes"]
        + 0.15 * out["verification_timeline_macro_f1"]
        + 0.30 * out["evidence_status_f1_yes"]
        + 0.35 * out["evidence_quality_macro_f1"]
    )
    return out

def print_metrics(m, title="metrics"):
    print("\n" + "="*80)
    print(title)
    print("="*80)
    for k, v in m.items():
        print(f"{k:36s}: {v:.6f}")

def adjust_probs(probs, multipliers):
    out = {}
    for t in TASKS:
        p = probs[t].copy()
        if t in multipliers:
            mult = np.array([multipliers[t].get(lab, 1.0) for lab in LABELS[t]], dtype=np.float32)
            p = p * mult.reshape(1, -1)
            p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p
    return out

def probs_to_pred_df(df, probs, config=None):
    if config is None:
        config = {}

    multipliers = config.get("multipliers", {})
    force_q_non_na = config.get("force_quality_non_na_when_evidence_yes", False)
    t1_threshold = config.get("t1_threshold", None)
    t3_threshold = config.get("t3_threshold", None)

    probs2 = adjust_probs(probs, multipliers)

    pred = pd.DataFrame()
    pred["id"] = df["id"].values

    if t1_threshold is not None:
        yes_id = LABEL2ID["promise_status"]["Yes"]
        pred["promise_status"] = np.where(probs2["promise_status"][:, yes_id] >= t1_threshold, "Yes", "No")
    else:
        ids = probs2["promise_status"].argmax(axis=1)
        pred["promise_status"] = [ID2LABEL["promise_status"][int(i)] for i in ids]

    ids = probs2["verification_timeline"].argmax(axis=1)
    pred["verification_timeline"] = [ID2LABEL["verification_timeline"][int(i)] for i in ids]

    if t3_threshold is not None:
        yes_id = LABEL2ID["evidence_status"]["Yes"]
        pred_es = []
        for row_probs in probs2["evidence_status"]:
            if row_probs[yes_id] >= t3_threshold:
                pred_es.append("Yes")
            else:
                na_id = LABEL2ID["evidence_status"]["N/A"]
                no_id = LABEL2ID["evidence_status"]["No"]
                pred_es.append("N/A" if row_probs[na_id] >= row_probs[no_id] else "No")
        pred["evidence_status"] = pred_es
    else:
        ids = probs2["evidence_status"].argmax(axis=1)
        pred["evidence_status"] = [ID2LABEL["evidence_status"][int(i)] for i in ids]

    ids = probs2["evidence_quality"].argmax(axis=1)
    pred["evidence_quality"] = [ID2LABEL["evidence_quality"][int(i)] for i in ids]

    pred = apply_hierarchy_to_pred_df(
        pred,
        force_quality_non_na_when_evidence_yes=force_q_non_na,
        probs=probs2,
    )
    return pred


## 4. 重建 Stage3C / Stage4A / stage4_conservative pipeline

In [ ]:

VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},
    "head_mid": {"head": 0.70, "middle": 0.30, "tail": 0.0},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},
    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "head_tail_more": {"head": 0.50, "middle": 0.10, "tail": 0.40},
    "head_mid_more": {"head": 0.50, "middle": 0.40, "tail": 0.10},
}

T2_VIEW_PRESETS = {
    "head": {"head": 1.0, "middle": 0.0, "tail": 0.0},
    "middle": {"head": 0.0, "middle": 1.0, "tail": 0.0},
    "tail": {"head": 0.0, "middle": 0.0, "tail": 1.0},
    "head_tail": {"head": 0.70, "middle": 0.0, "tail": 0.30},
    "mid_tail": {"head": 0.0, "middle": 0.50, "tail": 0.50},
    "even": {"head": 0.34, "middle": 0.33, "tail": 0.33},
    "head_mid_tail": {"head": 0.50, "middle": 0.25, "tail": 0.25},
    "tail_more": {"head": 0.30, "middle": 0.20, "tail": 0.50},
    "mid_more": {"head": 0.30, "middle": 0.50, "tail": 0.20},
}

def copy_obj(x):
    return json.loads(json.dumps(x))

def mix_views(stem_view_probs, split, task, preset_name):
    preset = VIEW_PRESETS[preset_name]
    out = None
    for view, w in preset.items():
        p = stem_view_probs[split][view][task]
        out = w * p if out is None else out + w * p
    return out

def build_tta_probs(split, config):
    weights = config.get("weights", {t: 0.5 for t in TASKS})
    view_presets = config.get("view_presets", {t: "head" for t in TASKS})

    out = {}
    for t in TASKS:
        preset_name = view_presets.get(t, "head")
        mac_p = mix_views(view_obj["mac"], split, t, preset_name)
        ckip_p = mix_views(view_obj["ckip"], split, t, preset_name)

        w_mac = weights.get(t, 0.5)
        p = w_mac * mac_p + (1 - w_mac) * ckip_p
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
        out[t] = p.astype(np.float32)

    return out

def eval_config_as_pred(df, split, config):
    probs = build_tta_probs(split, config)
    post_cfg = {k: v for k, v in config.items() if k not in ["weights", "view_presets"]}
    pred = probs_to_pred_df(df, probs, post_cfg)
    return pred, probs

def t4_adjust_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T4_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_quality_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    q = adjusted["evidence_quality"]

    non_na_ids = [
        LABEL2ID["evidence_quality"]["Clear"],
        LABEL2ID["evidence_quality"]["Not Clear"],
        LABEL2ID["evidence_quality"]["Misleading"],
    ]

    q3 = q[:, non_na_ids].copy()
    q3 = q3 / np.clip(q3.sum(axis=1, keepdims=True), 1e-12, None)
    return q3

def get_t4_blend_probs(base_probs, base_config, spec_probs, alpha=0.05, spec_multipliers=None):
    base_q3 = get_base_quality_non_na_probs(base_probs, base_config)
    spec_q3 = t4_adjust_probs(spec_probs, spec_multipliers=spec_multipliers)
    blend = (1 - alpha) * base_q3 + alpha * spec_q3
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)
    return blend.astype(np.float32)

def compose_quality_from_p3(base_pred, q3):
    out = base_pred.copy()
    mask = out["evidence_status"] == "Yes"
    ids = q3.argmax(axis=1)
    labels = np.array([T4_ID2LABEL[int(i)] for i in ids])
    out.loc[mask, "evidence_quality"] = labels[mask.values]
    out.loc[~mask, "evidence_quality"] = "N/A"
    return apply_hierarchy_to_pred_df(out)

def build_v34_base(df, split):
    base_pred, base_probs = eval_config_as_pred(df, split, V34_BASE_CONFIG)
    t4_probs = t4_val_probs if split == "val" else t4_test_probs
    q3 = get_t4_blend_probs(
        base_probs,
        V34_BASE_CONFIG,
        t4_probs,
        alpha=V34_T4_ALPHA,
        spec_multipliers=V34_T4_SPEC_MULT,
    )
    pred = compose_quality_from_p3(base_pred, q3)
    return pred, base_probs, q3

def mix_t2_views(split, preset_name):
    preset = T2_VIEW_PRESETS[preset_name]
    prob_dict = t2_obj[f"{split}_probs"]

    out = None
    for view, w in preset.items():
        p = prob_dict[view]
        out = w * p if out is None else out + w * p

    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def adjust_t2_spec_probs(spec_probs, spec_multipliers=None):
    p = spec_probs.copy()
    if spec_multipliers:
        mult = np.array([spec_multipliers.get(lab, 1.0) for lab in T2_LABELS], dtype=np.float32)
        p = p * mult.reshape(1, -1)
        p = p / np.clip(p.sum(axis=1, keepdims=True), 1e-12, None)
    return p

def get_base_timeline_non_na_probs(base_probs, base_config):
    multipliers = base_config.get("multipliers", {})
    adjusted = adjust_probs(base_probs, multipliers)
    p = adjusted["verification_timeline"]

    non_na_ids = [
        LABEL2ID["verification_timeline"]["already"],
        LABEL2ID["verification_timeline"]["within_2_years"],
        LABEL2ID["verification_timeline"]["between_2_and_5_years"],
        LABEL2ID["verification_timeline"]["more_than_5_years"],
    ]

    p4 = p[:, non_na_ids].copy()
    p4 = p4 / np.clip(p4.sum(axis=1, keepdims=True), 1e-12, None)
    return p4

def get_t2_blend_probs(split, base_probs, alpha, view_preset, spec_multipliers):
    base_p4 = get_base_timeline_non_na_probs(base_probs, V34_BASE_CONFIG)
    spec_p4 = mix_t2_views(split, view_preset)
    spec_p4 = adjust_t2_spec_probs(spec_p4, spec_multipliers=spec_multipliers)

    blend = (1 - alpha) * base_p4 + alpha * spec_p4
    blend = blend / np.clip(blend.sum(axis=1, keepdims=True), 1e-12, None)
    return blend.astype(np.float32)

def compose_timeline_from_p4(base_pred, p4):
    out = base_pred.copy()
    mask = out["promise_status"] == "Yes"
    ids = p4.argmax(axis=1)
    labels = np.array([T2_ID2LABEL[int(i)] for i in ids])
    out.loc[mask, "verification_timeline"] = labels[mask.values]
    out.loc[~mask, "verification_timeline"] = "N/A"
    return apply_hierarchy_to_pred_df(out)


In [ ]:

def normalize_t2_base_choice(base_choice):
    # Public Stage names are kept stable for README/output clarity. Some earlier
    # cells store conservative-search choices under the internal
    # stage4_conservative_* prefix, so map the public aliases here.
    aliases = {
        "stage3": "stage3",
        "v34": "stage3",  # backward-compatible alias for older Colab runs
        "stage4_distribution_matched": "stage4_conservative_distribution_matched",
        "stage4_safe": "stage4_conservative_safe",
        "stage4_very_safe": "stage4_conservative_very_safe",
        "stage4_conservative_distribution_matched": "stage4_conservative_distribution_matched",
        "stage4_conservative_best_val": "stage4_conservative_best_val",
        "stage4_conservative_safe": "stage4_conservative_safe",
        "stage4_conservative_very_safe": "stage4_conservative_very_safe",
        "stage4_t2_best_val": "stage4_t2_best_val",
    }
    return aliases.get(base_choice, base_choice)

def is_t2_base_choice_available(base_choice):
    # conservative search 可能找不到某些候選；不存在時後面就略過，避免整本 notebook 中斷。
    normalized = normalize_t2_base_choice(base_choice)
    if normalized in {"stage3", "stage4_t2_best_val"}:
        return True
    if normalized.startswith("stage4_conservative_"):
        name = normalized.replace("stage4_conservative_", "")
        return V35_01_CONFIG_OBJ.get("choices", {}).get(name) is not None
    return False

def filter_available_base_choices(candidates):
    available = []
    skipped = []
    for choice in candidates:
        if is_t2_base_choice_available(choice):
            available.append(choice)
        else:
            skipped.append(choice)
    if skipped:
        print("Skipping unavailable base choices:", skipped)
        print("If you need these choices, rerun the conservative search with looser thresholds.")
    if not available:
        raise RuntimeError("No available Stage 4 base choices. Check previous config cells first.")
    return available

def get_t2_cfg_from_choice(base_choice):
    original_choice = base_choice
    base_choice = normalize_t2_base_choice(base_choice)

    if base_choice == "stage4_t2_best_val":
        return {
            "view": V35_CONFIG_OBJ["best_view"],
            "alpha": V35_CONFIG_OBJ["best_alpha"],
            "spec_multipliers": V35_CONFIG_OBJ["best_spec_multipliers"],
        }

    if base_choice.startswith("stage4_conservative_"):
        name = base_choice.replace("stage4_conservative_", "")
        cfg = V35_01_CONFIG_OBJ.get("choices", {}).get(name)
        if cfg is None:
            raise ValueError(f"choice not found or None: {original_choice} -> {base_choice}")
        return {
            "view": cfg["view"],
            "alpha": cfg["alpha"],
            "spec_multipliers": cfg["spec_multipliers"],
        }

    if base_choice == "stage3":
        return {
            "view": "head",
            "alpha": 0.0,
            "spec_multipliers": None,
        }

    raise ValueError(f"unknown base_choice: {original_choice} -> {base_choice}")

def build_pipeline_probs_and_pred(df, split, base_choice):
    # Start from Stage 3 base: T1/T3 + T4 blend
    stage3_pred, base_probs, q3 = build_v34_base(df, split)

    # Add selected T2 blend
    t2_cfg = get_t2_cfg_from_choice(base_choice)
    p4 = get_t2_blend_probs(
        split=split,
        base_probs=base_probs,
        alpha=t2_cfg["alpha"],
        view_preset=t2_cfg["view"],
        spec_multipliers=t2_cfg["spec_multipliers"],
    )
    pred = compose_timeline_from_p4(stage3_pred, p4)

    # Re-apply quality from q3 after timeline composition
    pred = compose_quality_from_p3(pred, q3)

    return pred, base_probs, p4, q3, t2_cfg

BASE_CHOICE_CANDIDATES = ["stage3", "stage4_distribution_matched", "stage4_conservative_best_val", "stage4_t2_best_val"]
BASE_CHOICES = filter_available_base_choices(BASE_CHOICE_CANDIDATES)
print("Available base choices:", BASE_CHOICES)

for choice in BASE_CHOICES:
    pred, base_probs, p4, q3, cfg = build_pipeline_probs_and_pred(val_df, "val", choice)
    metrics = calc_metrics(val_df, pred)
    print_metrics(metrics, choice)
    print("t2_cfg:", cfg)
    print("timeline:", pred["verification_timeline"].value_counts().to_dict())
    print("quality:", pred["evidence_quality"].value_counts().to_dict())


## 5. Cue Feature Extraction

In [ ]:

def contains_any(text, patterns):
    return any(re.search(p, text) for p in patterns)

def count_any(text, patterns):
    return sum(1 for p in patterns if re.search(p, text))

T2_ALREADY_PATTERNS = [
    r"已完成", r"已取得", r"已通過", r"已導入", r"已建置", r"已建立", r"已執行", r"已實施",
    r"完成.{0,8}(查證|驗證|盤查|稽核|認證|確信)",
    r"通過.{0,8}(查證|驗證|認證|稽核)",
    r"取得.{0,8}(認證|證書|證明|查證)",
    r"於\s*\d{4}\s*年.{0,12}(完成|取得|通過|導入|建立|執行)",
]

T2_WITHIN2_PATTERNS = [
    r"兩年內", r"二年內", r"2\s*年內", r"短期", r"近期", r"明年", r"後年",
    r"2025\s*年?", r"2026\s*年?",
]

T2_BETWEEN25_PATTERNS = [
    r"未來五年", r"五年內", r"5\s*年內", r"中期", r"中程", r"階段性目標",
    r"2027\s*年?", r"2028\s*年?", r"2029\s*年?",
]

T2_MORE5_PATTERNS = [
    r"2030\s*年?", r"2035\s*年?", r"2040\s*年?", r"2050\s*年?",
    r"長期", r"長程", r"淨零", r"碳中和", r"RE100", r"SBTi", r"科學基礎減碳",
]

T4_CLEAR_PATTERNS = [
    r"第三方.{0,8}(查證|驗證|確信|認證)",
    r"(查證|驗證|確信)報告",
    r"ISO\s*\d+",
    r"取得.{0,8}(認證|證書|證明)",
    r"通過.{0,8}(認證|查證|驗證|稽核)",
    r"\d+(\.\d+)?\s*%",
    r"\d+(\.\d+)?\s*(公噸|噸|度|件|人|家|次|場|小時|萬元|億元|元|張|項)",
    r"100\s*%",
]

T4_NOT_CLEAR_PATTERNS = [
    r"持續推動", r"積極推動", r"積極落實", r"逐步", r"強化", r"提升", r"致力於",
    r"定期檢視", r"持續優化", r"持續改善", r"落實管理", r"完善", r"推廣",
    r"規劃", r"研擬", r"倡議", r"鼓勵",
]

T4_MISLEADING_PATTERNS = [
    r"宣稱", r"號稱", r"疑似", r"未說明", r"未揭露", r"未明確",
    r"相關資料不足", r"無法確認",
]

def extract_cues_one(text):
    text = str(text)
    compact = re.sub(r"\s+", "", text)

    years = [int(y) for y in re.findall(r"(20[2-5]\d)", compact)]
    future_year_max = max(years) if years else None
    future_year_min = min(years) if years else None

    has_numeric = bool(re.search(r"\d+(\.\d+)?", compact))
    has_percent = bool(re.search(r"\d+(\.\d+)?\s*%", compact))
    has_esg_cert = contains_any(compact, [r"ISO\s*\d+", r"第三方", r"查證", r"驗證", r"確信", r"認證", r"稽核"])

    cues = {
        "t2_already_cue": contains_any(compact, T2_ALREADY_PATTERNS),
        "t2_within2_cue": contains_any(compact, T2_WITHIN2_PATTERNS),
        "t2_between25_cue": contains_any(compact, T2_BETWEEN25_PATTERNS),
        "t2_more5_cue": contains_any(compact, T2_MORE5_PATTERNS),

        "t4_clear_cue": contains_any(compact, T4_CLEAR_PATTERNS) or (has_numeric and has_esg_cert),
        "t4_not_clear_cue": contains_any(compact, T4_NOT_CLEAR_PATTERNS),
        "t4_misleading_cue": contains_any(compact, T4_MISLEADING_PATTERNS),

        "has_numeric": has_numeric,
        "has_percent": has_percent,
        "has_esg_cert": has_esg_cert,
        "future_year_min": future_year_min if future_year_min is not None else -1,
        "future_year_max": future_year_max if future_year_max is not None else -1,
    }

    # 年份輔助：較遠年份推 more_than_5_years
    if future_year_max is not None:
        if future_year_max >= 2030:
            cues["t2_more5_cue"] = True
        elif 2027 <= future_year_max <= 2029:
            cues["t2_between25_cue"] = True
        elif 2025 <= future_year_max <= 2026:
            cues["t2_within2_cue"] = True

    # 如果文字同時有 already cue 和 long-term cue，通常是「已制定長期目標」或「已完成某事項、目標是2030」
    # 這類不要強硬決定，只交給 multiplier search。
    return cues

def build_cue_df(df):
    rows = []
    for text in df["data"].astype(str).tolist():
        rows.append(extract_cues_one(text))
    return pd.DataFrame(rows)

val_cues = build_cue_df(val_df)
test_cues = build_cue_df(test_df)

print("val cue rates:")
display(val_cues.mean(numeric_only=True).to_frame("mean").T)

print("test cue rates:")
display(test_cues.mean(numeric_only=True).to_frame("mean").T)

val_cues.to_csv(f"{OUT_DIR}/stage4_val_cue_features.csv", index=False, encoding="utf-8-sig")
test_cues.to_csv(f"{OUT_DIR}/stage4_test_cue_features.csv", index=False, encoding="utf-8-sig")


## 6. Cue multiplier presets

In [ ]:

T2_CUE_PRESETS = {
    "none": {
        "already": 1.00,
        "within2": 1.00,
        "between25": 1.00,
        "more5": 1.00,
    },
    "mild_t2": {
        "already": 1.05,
        "within2": 1.08,
        "between25": 1.06,
        "more5": 1.06,
    },
    "balanced_t2": {
        "already": 1.08,
        "within2": 1.15,
        "between25": 1.10,
        "more5": 1.10,
    },
    "timeline_years": {
        "already": 1.00,
        "within2": 1.20,
        "between25": 1.15,
        "more5": 1.18,
    },
    "already_guard": {
        "already": 1.12,
        "within2": 1.10,
        "between25": 1.06,
        "more5": 1.06,
    },
    "long_term": {
        "already": 1.00,
        "within2": 1.05,
        "between25": 1.10,
        "more5": 1.25,
    },
    "within2_focus": {
        "already": 1.00,
        "within2": 1.35,
        "between25": 1.00,
        "more5": 1.00,
    },
}

T4_CUE_PRESETS = {
    "none": {
        "clear": 1.00,
        "not_clear": 1.00,
        "misleading": 1.00,
    },
    "mild_t4": {
        "clear": 1.05,
        "not_clear": 1.04,
        "misleading": 1.00,
    },
    "clear_evidence": {
        "clear": 1.12,
        "not_clear": 1.00,
        "misleading": 1.00,
    },
    "not_clear_vague": {
        "clear": 1.00,
        "not_clear": 1.12,
        "misleading": 1.00,
    },
    "balanced_t4": {
        "clear": 1.08,
        "not_clear": 1.08,
        "misleading": 1.00,
    },
    "misleading_tiny": {
        "clear": 1.04,
        "not_clear": 1.06,
        "misleading": 1.10,
    },
}

def apply_t2_cues(p4, cues, preset_name):
    preset = T2_CUE_PRESETS[preset_name]
    out = p4.copy()

    mult = np.ones_like(out, dtype=np.float32)

    # T2_LABELS = already, within_2_years, between_2_and_5_years, more_than_5_years
    mult[:, T2_LABEL2ID["already"]] *= np.where(cues["t2_already_cue"].values, preset["already"], 1.0)
    mult[:, T2_LABEL2ID["within_2_years"]] *= np.where(cues["t2_within2_cue"].values, preset["within2"], 1.0)
    mult[:, T2_LABEL2ID["between_2_and_5_years"]] *= np.where(cues["t2_between25_cue"].values, preset["between25"], 1.0)
    mult[:, T2_LABEL2ID["more_than_5_years"]] *= np.where(cues["t2_more5_cue"].values, preset["more5"], 1.0)

    out = out * mult
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)

def apply_t4_cues(q3, cues, preset_name):
    preset = T4_CUE_PRESETS[preset_name]
    out = q3.copy()

    mult = np.ones_like(out, dtype=np.float32)

    # T4_LABELS = Clear, Not Clear, Misleading
    mult[:, T4_LABEL2ID["Clear"]] *= np.where(cues["t4_clear_cue"].values, preset["clear"], 1.0)
    mult[:, T4_LABEL2ID["Not Clear"]] *= np.where(cues["t4_not_clear_cue"].values, preset["not_clear"], 1.0)
    mult[:, T4_LABEL2ID["Misleading"]] *= np.where(cues["t4_misleading_cue"].values, preset["misleading"], 1.0)

    out = out * mult
    out = out / np.clip(out.sum(axis=1, keepdims=True), 1e-12, None)
    return out.astype(np.float32)


## 7. Cue calibration search

In [ ]:

def timeline_counts(pred):
    return {lab: int((pred["verification_timeline"] == lab).sum()) for lab in LABELS["verification_timeline"]}

def quality_counts(pred):
    return {lab: int((pred["evidence_quality"] == lab).sum()) for lab in LABELS["evidence_quality"]}

def calc_timeline_risk(test_pred, ref_dist):
    cnt = timeline_counts(test_pred)
    already_drop = max(0, ref_dist.get("already", 0) - cnt.get("already", 0))
    more5_inc = max(0, cnt.get("more_than_5_years", 0) - ref_dist.get("more_than_5_years", 0))
    within2_inc = max(0, cnt.get("within_2_years", 0) - ref_dist.get("within_2_years", 0))
    risk = already_drop / 120.0 + more5_inc / 120.0 + within2_inc / 25.0
    return float(risk), {
        "already_drop": int(already_drop),
        "more5_inc": int(more5_inc),
        "within2_inc": int(within2_inc),
    }

def calc_quality_risk(test_pred, ref_dist):
    cnt = quality_counts(test_pred)
    clear_inc = max(0, cnt.get("Clear", 0) - ref_dist.get("Clear", 0))
    not_clear_inc = max(0, cnt.get("Not Clear", 0) - ref_dist.get("Not Clear", 0))
    na_drop = max(0, ref_dist.get("N/A", 0) - cnt.get("N/A", 0))
    risk = clear_inc / 200.0 + not_clear_inc / 120.0 + na_drop / 120.0
    return float(risk), {
        "clear_inc": int(clear_inc),
        "not_clear_inc": int(not_clear_inc),
        "na_drop": int(na_drop),
    }

def build_cued_pred(df, split, base_choice, t2_cue_preset="none", t4_cue_preset="none"):
    cues = val_cues if split == "val" else test_cues

    base_pred, base_probs, p4, q3, t2_cfg = None, None, None, None, None

    # build selected base
    pred0, base_probs, p4, q3, t2_cfg = build_pipeline_probs_and_pred(df, split, base_choice)

    # apply cues to p4/q3
    p4_cued = apply_t2_cues(p4, cues, t2_cue_preset)
    q3_cued = apply_t4_cues(q3, cues, t4_cue_preset)

    pred = pred0.copy()
    pred = compose_timeline_from_p4(pred, p4_cued)
    pred = compose_quality_from_p3(pred, q3_cued)
    pred = apply_hierarchy_to_pred_df(pred)

    return pred, {
        "base_probs": base_probs,
        "p4_base": p4,
        "q3_base": q3,
        "p4_cued": p4_cued,
        "q3_cued": q3_cued,
        "t2_cfg": t2_cfg,
    }

# reference distributions
# reference distributions
stage3_test_pred, _, _, _, _ = build_pipeline_probs_and_pred(test_df, "test", "stage3")

ref_timeline_dist = timeline_counts(stage3_test_pred)
ref_quality_dist = quality_counts(stage3_test_pred)

print("ref timeline stage3:", ref_timeline_dist)
print("ref quality stage3:", ref_quality_dist)

search_rows = []
best_val = None
best_safe = None
best_dist = None
best_t2_only = None
best_t4_only = None

for base_choice in BASE_CHOICES:
    for t2_preset in T2_CUE_PRESETS.keys():
        for t4_preset in T4_CUE_PRESETS.keys():
            val_pred, _ = build_cued_pred(val_df, "val", base_choice, t2_preset, t4_preset)
            metrics = calc_metrics(val_df, val_pred)

            test_pred, _ = build_cued_pred(test_df, "test", base_choice, t2_preset, t4_preset)
            t_risk, t_parts = calc_timeline_risk(test_pred, ref_timeline_dist)
            q_risk, q_parts = calc_quality_risk(test_pred, ref_quality_dist)
            total_risk = t_risk + q_risk

            row = {
                "base_choice": base_choice,
                "t2_cue_preset": t2_preset,
                "t4_cue_preset": t4_preset,
                **metrics,
                "timeline_risk": t_risk,
                "quality_risk": q_risk,
                "total_risk": total_risk,
                **{f"timeline_{k}": v for k, v in timeline_counts(test_pred).items()},
                **{f"quality_{k}": v for k, v in quality_counts(test_pred).items()},
                **{f"risk_t_{k}": v for k, v in t_parts.items()},
                **{f"risk_q_{k}": v for k, v in q_parts.items()},
            }
            search_rows.append(row)

            item = (metrics["weighted_score"], metrics, val_pred, test_pred, base_choice, t2_preset, t4_preset, total_risk, t_risk, q_risk)

            if best_val is None or metrics["weighted_score"] > best_val[0]:
                best_val = item

            # safe: 要比 stage4_conservative distribution_matched 高或接近，且風險不要太高
            if metrics["weighted_score"] >= 0.6950 and total_risk <= 0.95:
                if best_safe is None or metrics["weighted_score"] > best_safe[0]:
                    best_safe = item

            # distribution matched: 至少比 v34 高，風險最小；同風險比分數
            if metrics["weighted_score"] >= 0.6945:
                if best_dist is None:
                    best_dist = item
                else:
                    if (total_risk < best_dist[7] - 1e-9) or (abs(total_risk - best_dist[7]) < 1e-9 and metrics["weighted_score"] > best_dist[0]):
                        best_dist = item

            # t2 only / t4 only diagnostic
            if t4_preset == "none" and t2_preset != "none":
                if best_t2_only is None or metrics["weighted_score"] > best_t2_only[0]:
                    best_t2_only = item

            if t2_preset == "none" and t4_preset != "none":
                if best_t4_only is None or metrics["weighted_score"] > best_t4_only[0]:
                    best_t4_only = item

search_df = pd.DataFrame(search_rows).sort_values(["weighted_score", "total_risk"], ascending=[False, True])
search_df.to_csv(f"{OUT_DIR}/stage4_cue_search.csv", index=False, encoding="utf-8-sig")

def show_item(name, item):
    print("\n" + "="*80)
    print(name)
    print("="*80)
    if item is None:
        print("None")
        return
    score, metrics, val_pred, test_pred, base_choice, t2_preset, t4_preset, total_risk, t_risk, q_risk = item
    print_metrics(metrics, name)
    print("base_choice:", base_choice)
    print("t2_cue_preset:", t2_preset)
    print("t4_cue_preset:", t4_preset)
    print("total_risk:", total_risk, "timeline_risk:", t_risk, "quality_risk:", q_risk)
    print("val timeline:", timeline_counts(val_pred))
    print("test timeline:", timeline_counts(test_pred))
    print("val quality:", quality_counts(val_pred))
    print("test quality:", quality_counts(test_pred))

show_item("Stage4C best_val", best_val)
show_item("Stage4C safe", best_safe)
show_item("Stage4C distribution_matched", best_dist)
show_item("Stage4C best_t2_only", best_t2_only)
show_item("Stage4C best_t4_only", best_t4_only)

display(search_df.head(30))
display(search_df.sort_values(["total_risk", "weighted_score"], ascending=[True, False]).head(30))


## 8. 儲存 val 分析與 config

In [ ]:

def save_val_outputs(pred, metrics, tag):
    val_full = pd.concat(
        [
            val_df[["id", "data", "company", "page_number"]].reset_index(drop=True),
            val_df[TASKS].add_prefix("true_").reset_index(drop=True),
            pred[TASKS].add_prefix("pred_").reset_index(drop=True),
            val_cues.add_prefix("cue_").reset_index(drop=True),
        ],
        axis=1,
    )

    for t in TASKS:
        val_full[f"err_{t}"] = val_full[f"true_{t}"] != val_full[f"pred_{t}"]

    val_full["any_error"] = val_full[[f"err_{t}" for t in TASKS]].any(axis=1)

    val_full.to_csv(f"{OUT_DIR}/{tag}_val1000_predictions.csv", index=False, encoding="utf-8-sig")
    val_full.to_excel(f"{OUT_DIR}/{tag}_val1000_error_analysis.xlsx", index=False)

    with pd.ExcelWriter(f"{OUT_DIR}/{tag}_val1000_confusion.xlsx", engine="openpyxl") as writer:
        for t in TASKS:
            cm = confusion_matrix(val_df[t], pred[t], labels=LABELS[t])
            cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in LABELS[t]], columns=[f"pred_{x}" for x in LABELS[t]])
            cm_df.to_excel(writer, sheet_name=t[:31])

    return val_full

def item_to_config(item):
    if item is None:
        return None
    score, metrics, val_pred, test_pred, base_choice, t2_preset, t4_preset, total_risk, t_risk, q_risk = item
    return {
        "base_choice": base_choice,
        "t2_cue_preset": t2_preset,
        "t4_cue_preset": t4_preset,
        "metrics": metrics,
        "total_risk": total_risk,
        "timeline_risk": t_risk,
        "quality_risk": q_risk,
        "val_timeline_dist": timeline_counts(val_pred),
        "test_timeline_dist": timeline_counts(test_pred),
        "val_quality_dist": quality_counts(val_pred),
        "test_quality_dist": quality_counts(test_pred),
    }

choices = {
    "best_val": best_val,
    "safe": best_safe,
    "distribution_matched": best_dist,
    "best_t2_only": best_t2_only,
    "best_t4_only": best_t4_only,
}

config_obj = {
    "base": "stage4_t2_specialist_calibration",
    "t2_cue_presets": T2_CUE_PRESETS,
    "t4_cue_presets": T4_CUE_PRESETS,
    "reference": {
        "v34_test_timeline": ref_timeline_dist,
        "v34_test_quality": ref_quality_dist,
    },
    "choices": {name: item_to_config(item) for name, item in choices.items()},
}

with open(f"{OUT_DIR}/stage4_config.json", "w", encoding="utf-8") as f:
    json.dump(config_obj, f, ensure_ascii=False, indent=2)

for name, item in choices.items():
    if item is not None:
        _, metrics, val_pred, _, _, _, _, _, _, _ = item
        save_val_outputs(val_pred, metrics, f"stage4_{name}")

print("saved config and val outputs to:", OUT_DIR)
print(json.dumps(config_obj, ensure_ascii=False, indent=2)[:7000])


## 9. 產生 test submissions

In [ ]:

def compact_text(s):
    return re.sub(r"\s+", "", str(s))

train_map = {}
for _, row in train_df.iterrows():
    train_map[compact_text(row["data"])] = {t: row[t] for t in TASKS}

def make_submission_from_pred(pred, tag):
    pred = pred.copy()

    override_count = 0
    for i, row in test_df.iterrows():
        key = compact_text(row["data"])
        if key in train_map:
            for t in TASKS:
                pred.loc[i, t] = train_map[key][t]
            override_count += 1

    pred = apply_hierarchy_to_pred_df(pred)
    sub = pred[["id"] + TASKS].copy()

    path = f"{OUT_DIR}/{tag}_test2000_submission.csv"
    sub.to_csv(path, index=False, encoding="utf-8-sig")

    print("\n", tag, "saved:", path)
    print("override_count:", override_count)
    for t in TASKS:
        print(t, sub[t].value_counts().to_dict())

    return sub, path

submission_subs = {}
submission_paths = {}

# diagnostic bases：只產生目前 config 裡真的存在的候選。
DIAGNOSTIC_BASE_CHOICES = filter_available_base_choices(["stage3", "stage4_distribution_matched", "stage4_t2_best_val"])
for base_choice in DIAGNOSTIC_BASE_CHOICES:
    pred, _, _, _, _ = build_pipeline_probs_and_pred(test_df, "test", base_choice)
    sub, path = make_submission_from_pred(pred, f"stage4_base_{base_choice}")
    submission_subs[f"base_{base_choice}"] = sub
    submission_paths[f"base_{base_choice}"] = path

for name, item in choices.items():
    if item is None:
        continue
    _, metrics, val_pred, test_pred, base_choice, t2_preset, t4_preset, total_risk, t_risk, q_risk = item
    sub, path = make_submission_from_pred(test_pred, f"stage4_{name}")
    submission_subs[name] = sub
    submission_paths[name] = path

print("\nSubmission paths:")
for name, path in submission_paths.items():
    print(name, ":", path)


## 10. 報告摘要

In [ ]:

report = []
report.append("# Stage 4 T2 Specialist ? Cue Calibration ??")
report.append("")
report.append("## Cue preset search choices")

for name, item in choices.items():
    report.append(f"### {name}")
    cfg = item_to_config(item)
    if cfg is None:
        report.append("- None")
    else:
        report.append(f"- base_choice: {cfg['base_choice']}")
        report.append(f"- t2_cue_preset: {cfg['t2_cue_preset']}")
        report.append(f"- t4_cue_preset: {cfg['t4_cue_preset']}")
        report.append(f"- total_risk: {cfg['total_risk']:.6f}")
        report.append(f"- timeline_risk: {cfg['timeline_risk']:.6f}")
        report.append(f"- quality_risk: {cfg['quality_risk']:.6f}")
        for k, v in cfg["metrics"].items():
            report.append(f"- {k}: {v:.6f}")
        report.append(f"- val_timeline_dist: {cfg['val_timeline_dist']}")
        report.append(f"- test_timeline_dist: {cfg['test_timeline_dist']}")
        report.append(f"- val_quality_dist: {cfg['val_quality_dist']}")
        report.append(f"- test_quality_dist: {cfg['test_quality_dist']}")

report.append("")
report.append("## Submission distributions")
for name, sub in submission_subs.items():
    report.append(f"### {name}")
    for t in TASKS:
        report.append(f"#### {t}")
        for lab, cnt in sub[t].value_counts().to_dict().items():
            report.append(f"- {lab}: {cnt}")

report_md = "\n".join(report)

with open(f"{OUT_DIR}/stage4_report.md", "w", encoding="utf-8") as f:
    f.write(report_md)

print(report_md[:8000])
print("saved:", f"{OUT_DIR}/stage4_report.md")
